In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    max,
    when,
    lit,
    coalesce,
    current_timestamp
)

In [0]:
customers = spark.table("silver_customers")
addresses = spark.table("silver_addresses")
orders = spark.table("silver_orders")
payments = spark.table("silver_payments")
tickets = spark.table("silver_tickets")
loyalty = spark.table("silver_loyalty")

In [0]:
customer_base = customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "gender",
    "status",
    "signup_date"
)

In [0]:
address_summary = (
    addresses
    .filter(col("is_primary") == "true")
    .select(
        "customer_id",
        "state",
        "postcode",
        "country"
    ).dropDuplicates(["customer_id"])
)

In [0]:
REVENUE_STATUSES = ["PAID", "SHIPPED", "DELIVERED"]
order_summary = (
    orders.groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum(
            when(
                col("order_status").isin(REVENUE_STATUSES),
                col("order_amount")
            ).otherwise(lit(0))
        ).alias("total_revenue"),
        max("order_date").alias("last_order_date")
    )
)

In [0]:
payment_summary = (
    payments.groupBy("customer_id")
    .agg(
        count(
            when(
                col("payment_status") == "SUCCESS", 
                col("payment_id")
            )
        ).alias("successful_payment_count"),
        sum(
            when(
                col("payment_status") == "SUCCESS", 
                col("payment_amount")
            ).otherwise(lit(0))
        ).alias("successful_payment_amount"),
        max("payment_date").alias("last_payment_date")
    )
)

In [0]:
support_summary = (
    tickets.groupBy("customer_id")
    .agg(
        count("ticket_id").alias("support_ticket_count"),
        count(
            when(
                col("ticket_status").isin(["OPEN", "IN_PROGRESS"]),
                col("ticket_id")
            )
        ).alias("open_ticket_count"),
        max("ticket_created_at").alias("last_support_ticket_date")
    )
)

In [0]:
loyalty_summary = (
    loyalty
    .select(
        "customer_id",
        col("tier").alias("loyalty_tier"),
        "points_balance",
        "loyalty_status"
    )
    .dropDuplicates(["customer_id"])
)

In [0]:
customer_360 = (
    customer_base
    .join(address_summary, on="customer_id", how="left")
    .join(order_summary, on="customer_id", how="left")
    .join(payment_summary, on="customer_id", how="left")
    .join(support_summary, on="customer_id", how="left")
    .join(loyalty_summary, on="customer_id", how="left")
    .withColumn("total_orders", coalesce(col("total_orders"), lit(0)))
    .withColumn("total_revenue", coalesce(col("total_revenue"), lit(0.0)))
    .withColumn("successful_payment_count", coalesce(col("successful_payment_count"), lit(0)))
    .withColumn("successful_payment_amount", coalesce(col("successful_payment_amount"), lit(0.0)))
    .withColumn("support_ticket_count", coalesce(col("support_ticket_count"), lit(0)))
    .withColumn("open_ticket_count", coalesce(col("open_ticket_count"), lit(0)))
    .withColumn("points_balance", coalesce(col("points_balance"), lit(0)))
    .withColumn("_gold_processed_at", current_timestamp())
)

In [0]:
(
    customer_360
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_customer_360")
)

In [0]:
print("Customer base rows:", customer_base.count())
print("Gold customer 360 rows:", customer_360.count())

Customer base rows: 99048
Gold customer 360 rows: 99048
